# contract-eval quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianfoerste/contract-review-eval-harness/blob/main/examples/quickstart.ipynb)

Score an AI contract review against a hand-authored answer set, then let the release
policy decide it. Everything runs offline on synthetic contracts: no API key, no model call.

The package is not on PyPI yet. The wheel carries the bundled cases, so installing it
straight from GitHub is enough.

In [ ]:
%pip install --quiet "contract-eval @ git+https://github.com/sebastianfoerste/contract-review-eval-harness"

## Score the bundled stub review of the NDA

The stub is the harness's deterministic baseline. It deliberately over-flags one clause
and quotes one sentence that is not in the contract, so the scorecard always shows an
imperfect review being caught.

In [ ]:
from contract_eval import ReviewOutput, evaluate_review, load_case
from contract_eval.paths import fixture_file, project_root

case = load_case("nda")
stub_path = project_root() / fixture_file("nda")
stub = ReviewOutput.model_validate_json(stub_path.read_text(encoding="utf-8"))

result = evaluate_review(stub, case)
print("verdict:", result.verdict)
print("blockers:", result.blockers)
for name in ("clause_f1", "risk_f1", "risk_severity_accuracy", "span_coverage", "citation_grounding"):
    print(f"{name}: {result.scores[name]:.3f}")
print("hallucination_count:", result.scores["hallucination_count"])

## Drop the fabricated citation and score again

A quote counts as grounded only when it is an exact span of the normalised contract.
Keeping only grounded citations removes the hallucination and with it the blockers.
The over-flag and the severity mistakes remain, so the review still needs a human.

In [ ]:
from contract_eval.scorer import citation_grounding

grounded = [c for c in stub.citations if citation_grounding(case.source_text, [c]).grounded]
dropped = [c.clause_type for c in stub.citations if c not in grounded]
tweaked = stub.model_copy(update={"citations": grounded})

retry = evaluate_review(tweaked, case)
print("dropped citations for:", dropped)
print("verdict:", result.verdict, "->", retry.verdict)
print("hallucination_count:", result.scores["hallucination_count"], "->", retry.scores["hallucination_count"])
print("still to review:", retry.review_items)

## Next

Score your own tool's output with `evaluate_review(your_review, load_case("nda"))`, or gate
it in pytest with `contract_eval.pytest_plugin.assert_review_passes`; see
`examples/test_my_reviewer.py` in the repository.